<a href="https://colab.research.google.com/github/tanishjain-tmj/Loan-Approval-Prediction-ML/blob/main/Loan_Approval_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Loan Approval Prediction

In [34]:
# importing the Dependencies(libraries)

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import accuracy_score

In [35]:
# loading the loan approval dataset
loan_df = pd.read_csv('/content/loan_data.csv')

In [36]:
#exploring the dataset
loan_df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
1,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
2,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
3,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
4,LP001013,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360.0,1.0,Urban,Y


In [37]:
loan_df.shape

(381, 13)

In [38]:
loan_df.describe().T

,count,mean,std,min,25%,50%,75%,max
ApplicantIncome,381.0,3579.845144,1419.813818,150.0,2600.0,3333.0,4288.0,9703.0
CoapplicantIncome,381.0,1277.275381,2340.818114,0.0,0.0,983.0,2016.0,33837.0
LoanAmount,381.0,104.986877,28.358464,9.0,90.0,110.0,127.0,150.0
Loan_Amount_Term,370.0,340.864865,68.549257,12.0,360.0,360.0,360.0,480.0
Credit_History,351.0,0.837607,0.369338,0.0,1.0,1.0,1.0,1.0


In [39]:
# checking the missing values in the dataset
loan_df.isnull().sum()

,0
Loan_ID,0
Gender,5
Married,0
Dependents,8
Education,0
Self_Employed,21
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,11


In [40]:
# filling missing values
loan_df['Gender'] = loan_df['Gender'].fillna(loan_df['Gender'].mode()[0])

loan_df['Dependents'] = loan_df['Dependents'].fillna(loan_df['Dependents'].mode()[0])

loan_df['Self_Employed'] = loan_df['Self_Employed'].fillna(loan_df['Self_Employed'].mode()[0])

loan_df['Loan_Amount_Term'] = loan_df['Loan_Amount_Term'].fillna(loan_df['Loan_Amount_Term'].mode()[0])

loan_df['Credit_History'] = loan_df['Credit_History'].fillna(loan_df['Credit_History'].mode()[0])

In [41]:
loan_df.isnull().sum()

,0
Loan_ID,0
Gender,0
Married,0
Dependents,0
Education,0
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,0


In [42]:
# checking the distribution of Loan_Status
loan_df['Loan_Status'].value_counts()

,count
Loan_Status,
Y,271
N,110


In [43]:
# getting the mean values according to Loan_Status
loan_df.groupby('Loan_Status')[['ApplicantIncome',
                                    'CoapplicantIncome',
                                    'LoanAmount']].mean()

,ApplicantIncome,CoapplicantIncome,LoanAmount
Loan_Status,,,
N,3602.472727,1244.190909,103.154545
Y,3570.660517,1290.704502,105.730627


In [44]:
# checking the categorical columns
loan_df.dtypes

,0
Loan_ID,object
Gender,object
Married,object
Dependents,object
Education,object
Self_Employed,object
ApplicantIncome,int64
CoapplicantIncome,float64
LoanAmount,float64
Loan_Amount_Term,float64


In [45]:
# converting categorical values into numerical values
loan_df['Gender'] = loan_df['Gender'].map({'Male': 1, 'Female': 0})
loan_df['Married'] = loan_df['Married'].map({'Yes': 1, 'No': 0})
loan_df['Dependents'] = loan_df['Dependents'].replace('3+', 3)
loan_df['Dependents'] = loan_df['Dependents'].astype(int)
loan_df['Education'] = loan_df['Education'].map({
    'Graduate': 1,
    'Not Graduate': 0
})
loan_df['Self_Employed'] = loan_df['Self_Employed'].map({
    'Yes': 1,
    'No': 0
})
loan_df['Property_Area'] = loan_df['Property_Area'].map({
    'Urban': 2,
    'Semiurban': 1,
    'Rural': 0
})
loan_df['Loan_Status'] = loan_df['Loan_Status'].map({
    'Y': 1,
    'N': 0
})

In [46]:
# separating the data and label
X = loan_df.drop(columns=['Loan_ID', 'Loan_Status'])
Y = loan_df['Loan_Status']

In [47]:
X.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,1,1,1,1,0,4583,1508.0,128.0,360.0,1.0,0
1,1,1,0,1,1,3000,0.0,66.0,360.0,1.0,2
2,1,1,0,0,0,2583,2358.0,120.0,360.0,1.0,2
3,1,0,0,1,0,6000,0.0,141.0,360.0,1.0,2
4,1,1,0,0,0,2333,1516.0,95.0,360.0,1.0,2


In [48]:
Y.head()

,Loan_Status
0,0
1,1
2,1
3,1
4,1


In [49]:
print(X.shape)
print(Y.shape)

(381, 11)
(381,)


In [50]:
# standardizing the data
scaler = StandardScaler()
scaler.fit(X)
standardized_data = scaler.transform(X)

In [51]:
X = standardized_data

In [52]:
print(X)

[[ 0.53587514  0.81917802  0.33794768 ...  0.27514748  0.41943525
  -1.35183217]
 [ 0.53587514  0.81917802 -0.67589536 ...  0.27514748  0.41943525
   1.21698607]
 [ 0.53587514  0.81917802 -0.67589536 ...  0.27514748  0.41943525
   1.21698607]
 ...
 [-1.86610636 -1.22073588 -0.67589536 ...  0.27514748  0.41943525
  -1.35183217]
 [ 0.53587514  0.81917802  2.36563375 ... -2.39005229  0.41943525
  -1.35183217]
 [-1.86610636 -1.22073588 -0.67589536 ...  0.27514748 -2.38415824
  -0.06742305]]


In [53]:
# train test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=2
)

In [54]:
print(X.shape, X_train.shape, X_test.shape)

(381, 11) (304, 11) (77, 11)


In [55]:
print(Y.shape, Y_train.shape, Y_test.shape)

(381,) (304,) (77,)


In [56]:
# training the SVM classifier
classifier = svm.SVC(kernel='linear')

# training the SVM classifier
classifier.fit(X_train, Y_train)

SVC(kernel='linear')

In [57]:
# accuracy score on the training data

X_train_prediction = classifier.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.8453947368421053


In [58]:
# accuracy score on the test data

X_test_prediction = classifier.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.8441558441558441


In [59]:
# making a predictive system

input_data = (1, 1, 0, 1, 0, 5000, 2000, 150, 360, 1, 2)

# changing the input data to a numpy array
input_data_as_numpy_array = np.asarray(input_data)

# reshape the numpy array as we are predicting for one instance
input_data_reshaped = input_data_as_numpy_array.reshape(1, -1)

# standardize the input data
std_data = scaler.transform(input_data_reshaped)
print(std_data)
prediction = classifier.predict(std_data)
print(prediction)

if prediction[0] == 1:
    print('Loan is Approved')
else:
    print('Loan is Not Approved')

[[ 0.53587514  0.81917802 -0.67589536  0.60869007 -0.31805042  1.00155544
   0.30915471  1.58937779  0.27514748  0.41943525  1.21698607]]
[1]
Loan is Approved


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
